In [1]:
# ======================================
# Feature Engineering V4 - Setup
# ======================================
#
# Purpose:
# --------
# Build the V4 feature table for AlphaEngine.
#
# V4 adds:
#   1. Explicit ranked valuation features
#   2. Sector-relative valuation features
#   3. Core Alpha composite features
#   4. Convex Alpha composite features
#   5. Optional historical market-cap features if a clean panel exists
#
# Important:
#   Do NOT use current yfinance market cap as a historical feature.
#   Market cap must be point-in-time / historical to avoid leakage.

from pathlib import Path
import json
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_DIR = DATA_DIR / "raw"
UNIVERSE_DIR = DATA_DIR / "universe"

FEATURE_FILE_V2 = PROCESSED_DIR / "features_v2.parquet"
FEATURE_FILE_V4 = PROCESSED_DIR / "features_v4.parquet"
FEATURE_FAMILY_FILE_V4 = PROCESSED_DIR / "feature_families_v4.json"

UNIVERSE_METADATA_FILE = UNIVERSE_DIR / "sp500_metadata.csv"

# Optional future file.
# Expected columns if available:
#   date | ticker | market_cap
#
# This should be historical / point-in-time safe.
MARKET_CAP_FILE = PROCESSED_DIR / "market_cap_history.parquet"

df = pd.read_parquet(FEATURE_FILE_V2)

df["date"] = pd.to_datetime(df["date"])
df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()

# ---- Optional metadata merge ----
# If features_v2 already has sector, this will leave it alone.
# If it does not, use the S&P 500 metadata created in notebook 01.
if UNIVERSE_METADATA_FILE.exists():
    metadata = pd.read_csv(UNIVERSE_METADATA_FILE)

    metadata["ticker"] = (
        metadata["ticker"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    metadata_cols = [
        c for c in [
            "ticker",
            "company",
            "sector",
            "sub_industry",
            "headquarters",
            "date_added",
            "cik",
            "founded",
        ]
        if c in metadata.columns
    ]

    metadata = metadata[metadata_cols].drop_duplicates(subset=["ticker"])

    # Only merge metadata columns that are not already present.
    cols_to_add = [
        c for c in metadata.columns
        if c != "ticker" and c not in df.columns
    ]

    if cols_to_add:
        df = df.merge(
            metadata[["ticker"] + cols_to_add],
            on="ticker",
            how="left",
        )

        print("Merged metadata columns:", cols_to_add)
    else:
        print("No new metadata columns needed from:", UNIVERSE_METADATA_FILE)

else:
    print("No universe metadata file found:", UNIVERSE_METADATA_FILE)

df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

print("Loaded V2 features:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Tickers:", df["ticker"].nunique())

df.head()

Merged metadata columns: ['company', 'sub_industry', 'headquarters', 'date_added', 'cik', 'founded']
Loaded V2 features: (1903828, 15)
Date range: 2010-01-04 00:00:00 to 2025-12-31 00:00:00
Tickers: 503


,date,ticker,ret_6m,ret_12m,vol_12m,drawdown,pe_ratio,earnings_yield,sector,company,sub_industry,headquarters,date_added,cik,founded
0,2010-01-04,A,NaN,NaN,NaN,0.000000,26.902868,0.037171,Healthcare,Agilent Technologies,Life Sciences Tools & Services,"Santa Clara, California",2000-06-05,1090872.0,1999
1,2010-01-05,A,NaN,NaN,NaN,-0.010863,26.902868,0.037171,Healthcare,Agilent Technologies,Life Sciences Tools & Services,"Santa Clara, California",2000-06-05,1090872.0,1999
2,2010-01-06,A,NaN,NaN,NaN,-0.014377,26.902868,0.037171,Healthcare,Agilent Technologies,Life Sciences Tools & Services,"Santa Clara, California",2000-06-05,1090872.0,1999
3,2010-01-07,A,NaN,NaN,NaN,-0.015655,26.902868,0.037171,Healthcare,Agilent Technologies,Life Sciences Tools & Services,"Santa Clara, California",2000-06-05,1090872.0,1999
4,2010-01-08,A,NaN,NaN,NaN,-0.015974,26.902868,0.037171,Healthcare,Agilent Technologies,Life Sciences Tools & Services,"Santa Clara, California",2000-06-05,1090872.0,1999


In [2]:
(df.notnull().sum() / len(df) * 100)

date              100.000000
ticker            100.000000
ret_6m             96.675225
ret_12m            93.354442
vol_12m            93.354442
drawdown          100.000000
pe_ratio           94.588114
earnings_yield    100.000000
sector             99.788689
company            98.548241
sub_industry       98.548241
headquarters       98.548241
date_added         98.548241
cik                98.548241
founded            98.548241
dtype: float64

In [3]:
# ======================================
# Input Validation
# ======================================

required_base_cols = [
    "date",
    "ticker",
    "ret_6m",
    "ret_12m",
    "vol_12m",
    "drawdown",
    "pe_ratio",
    "earnings_yield",
    "sector",
]

missing_required_cols = [
    c for c in required_base_cols
    if c not in df.columns
]

if missing_required_cols:
    raise ValueError(f"Missing required input columns: {missing_required_cols}")

print("Required input columns present.")

print("\nInput coverage:")
input_coverage = (
    df[required_base_cols]
    .notna()
    .mean()
    .sort_values()
    .to_frame("non_null_pct")
)

input_coverage

Required input columns present.

Input coverage:


,non_null_pct
ret_12m,0.933544
vol_12m,0.933544
pe_ratio,0.945881
ret_6m,0.966752
sector,0.997887
date,1.000000
ticker,1.000000
drawdown,1.000000
earnings_yield,1.000000


In [4]:
# ======================================
# Cross-Sectional Rank Features
# ======================================
#
# These are shared by Core Alpha and Convex Alpha.
#
# Rank convention:
#   Higher rank = "more" of that attribute.
#
# Examples:
#   ret_12m_rank: higher = stronger 12M momentum
#   vol_12m_rank: higher = more volatile
#   low_vol_rank: higher = lower volatility
#   earnings_yield_rank: higher = cheaper / better earnings yield
#   pe_cheap_rank: higher = lower P/E / cheaper
#
# Drawdown convention:
#   drawdown_rank: higher = less drawdown / more stable
#   drawdown_severity_rank: higher = deeper drawdown / more rebound potential

# ---- Momentum / risk ranks ----
df["ret_6m_rank"] = df.groupby("date")["ret_6m"].rank(pct=True)
df["ret_12m_rank"] = df.groupby("date")["ret_12m"].rank(pct=True)
df["vol_12m_rank"] = df.groupby("date")["vol_12m"].rank(pct=True)
df["drawdown_rank"] = df.groupby("date")["drawdown"].rank(pct=True)

# Lower volatility = higher low-vol rank
df["low_vol_rank"] = 1.0 - df["vol_12m_rank"]

# Higher drawdown severity = more beaten down
df["drawdown_severity_rank"] = 1.0 - df["drawdown_rank"]

# ---- Valuation ranks ----
df["earnings_yield_rank"] = (
    df.groupby("date")["earnings_yield"]
    .rank(pct=True)
)

# Backward-compatible alias from V3.
# Existing notebooks may still refer to ey_rank.
df["ey_rank"] = df["earnings_yield_rank"]

# Higher P/E = more expensive
df["pe_ratio_rank"] = (
    df.groupby("date")["pe_ratio"]
    .rank(pct=True)
)

# Lower P/E = cheaper
df["pe_cheap_rank"] = 1.0 - df["pe_ratio_rank"]

# ---- Sector-relative valuation ranks ----
df["sector_earnings_yield_rank"] = (
    df.groupby(["date", "sector"])["earnings_yield"]
    .rank(pct=True)
)

df["sector_pe_ratio_rank"] = (
    df.groupby(["date", "sector"])["pe_ratio"]
    .rank(pct=True)
)

df["sector_pe_cheap_rank"] = 1.0 - df["sector_pe_ratio_rank"]

# ---- Valuation composites ----
df["valuation_rank_combo"] = (
    df["earnings_yield_rank"]
    + df["pe_cheap_rank"]
)

df["sector_valuation_rank_combo"] = (
    df["sector_earnings_yield_rank"]
    + df["sector_pe_cheap_rank"]
)

# V3 quality/value combo:
#   earnings yield rank + low volatility rank
df["quality_value_combo"] = (
    df["earnings_yield_rank"]
    + df["low_vol_rank"]
)

# Explicitly named V4 version for clarity.
df["quality_value_combo_ranked"] = (
    df["earnings_yield_rank"]
    + df["low_vol_rank"]
)

print("Added V4 cross-sectional rank and valuation features")

Added V4 cross-sectional rank and valuation features


In [5]:
(df.notnull().sum() / len(df) * 100)

date                           100.000000
ticker                         100.000000
ret_6m                          96.675225
ret_12m                         93.354442
vol_12m                         93.354442
drawdown                       100.000000
pe_ratio                        94.588114
earnings_yield                 100.000000
sector                          99.788689
company                         98.548241
sub_industry                    98.548241
headquarters                    98.548241
date_added                      98.548241
cik                             98.548241
founded                         98.548241
ret_6m_rank                     96.675225
ret_12m_rank                    93.354442
vol_12m_rank                    93.354442
drawdown_rank                  100.000000
low_vol_rank                    93.354442
drawdown_severity_rank         100.000000
earnings_yield_rank            100.000000
ey_rank                        100.000000
pe_ratio_rank                   94

In [6]:
# ======================================
# Optional Historical Market Cap Features
# ======================================
#
# Market cap is extremely useful, but only if it is historical / point-in-time safe.
#
# Expected market cap panel:
#   date | ticker | market_cap
#
# If MARKET_CAP_FILE does not exist, this cell skips size features.
# Do NOT substitute current market cap across all historical dates.

HAS_MARKET_CAP = MARKET_CAP_FILE.exists()

if HAS_MARKET_CAP:
    market_cap_df = pd.read_parquet(MARKET_CAP_FILE)

    market_cap_df["date"] = pd.to_datetime(market_cap_df["date"])
    market_cap_df = market_cap_df.sort_values(["ticker", "date"]).reset_index(drop=True)

    required_cols = {"date", "ticker", "market_cap"}
    missing_cols = required_cols - set(market_cap_df.columns)

    if missing_cols:
        raise ValueError(f"Market cap file missing columns: {missing_cols}")

    df = df.merge(
        market_cap_df[["date", "ticker", "market_cap"]],
        on=["date", "ticker"],
        how="left",
    )

    df["log_market_cap"] = np.log1p(df["market_cap"])

    df["market_cap_rank"] = (
        df.groupby("date")["market_cap"]
        .rank(pct=True)
    )

    df["sector_market_cap_rank"] = (
        df.groupby(["date", "sector"])["market_cap"]
        .rank(pct=True)
    )

    # Convex framing:
    #   Higher room = smaller relative size / more room to scale.
    df["market_cap_room"] = 1.0 - df["market_cap_rank"]
    df["sector_market_cap_room"] = 1.0 - df["sector_market_cap_rank"]

    print("Added historical market cap features")
    print("Market cap coverage:", df["market_cap"].notna().mean())

else:
    print("No historical market cap file found.")
    print("Skipping size features for now:", MARKET_CAP_FILE)

No historical market cap file found.
Skipping size features for now: /Users/neilyejjey/stock_signal_engine_v1/data/processed/market_cap_history.parquet


In [7]:
# ======================================
# Risk-Adjusted Momentum
# ======================================

df["momentum_vol_adj"] = df["ret_12m"] / (df["vol_12m"] + 1e-6)

print("Added momentum_vol_adj")

Added momentum_vol_adj


In [8]:
(df.notnull().sum() / len(df) * 100)

date                           100.000000
ticker                         100.000000
ret_6m                          96.675225
ret_12m                         93.354442
vol_12m                         93.354442
drawdown                       100.000000
pe_ratio                        94.588114
earnings_yield                 100.000000
sector                          99.788689
company                         98.548241
sub_industry                    98.548241
headquarters                    98.548241
date_added                      98.548241
cik                             98.548241
founded                         98.548241
ret_6m_rank                     96.675225
ret_12m_rank                    93.354442
vol_12m_rank                    93.354442
drawdown_rank                  100.000000
low_vol_rank                    93.354442
drawdown_severity_rank         100.000000
earnings_yield_rank            100.000000
ey_rank                        100.000000
pe_ratio_rank                   94

In [9]:
# ======================================
# Shared, Core, and Convex Composite Features
# ======================================
#
# Shared:
#   General ranking / signal features useful for both model families.
#
# Core Alpha:
#   Wants stable, established, reasonably valued outperformers.
#
# Convex Alpha:
#   Wants right-tail candidates with momentum, optionality, and room to scale.

# ---- Shared momentum composite ----
df["momentum_composite"] = (
    df["ret_6m_rank"]
    + df["ret_12m_rank"]
) / 2


# ======================================
# Core Alpha Features
# ======================================
#
# Core Alpha thesis:
#   Find durable VOO beaters.
#
# Emphasis:
#   - quality/value
#   - lower volatility
#   - sector-relative valuation
#   - stability

df["core_quality_value_combo"] = (
    df["sector_valuation_rank_combo"]
    + df["low_vol_rank"]
)

df["core_stability_combo"] = (
    df["low_vol_rank"]
    + df["drawdown_rank"]
)

df["core_momentum_quality_combo"] = (
    df["momentum_composite"]
    + df["quality_value_combo_ranked"]
)

df["core_defensive_value_combo"] = (
    df["low_vol_rank"]
    + df["sector_earnings_yield_rank"]
)


# If historical market cap exists, add size-aware Core features.
if HAS_MARKET_CAP:
    df["core_size_quality_combo"] = (
        df["market_cap_rank"]
        + df["quality_value_combo_ranked"]
    )

    df["core_stability_size_combo"] = (
        df["market_cap_rank"]
        + df["low_vol_rank"]
    )

    df["core_large_cap_value_combo"] = (
        df["market_cap_rank"]
        + df["sector_valuation_rank_combo"]
    )


# ======================================
# Convex Alpha Features
# ======================================
#
# Convex Alpha thesis:
#   Find public-market power-law candidates.
#
# Emphasis:
#   - momentum
#   - optionality / volatility
#   - valuation support
#   - rebound potential
#   - room to scale, if market cap exists

df["convex_momentum_value_combo"] = (
    df["ret_12m_rank"]
    + df["valuation_rank_combo"]
)

df["convex_momentum_vol_combo"] = (
    df["ret_12m_rank"]
    + df["vol_12m_rank"]
)

# Rebound uses drawdown severity, not drawdown stability.
df["convex_rebound_combo"] = (
    df["drawdown_severity_rank"]
    + df["ret_6m_rank"]
)

df["convex_value_rebound_combo"] = (
    df["valuation_rank_combo"]
    + df["drawdown_severity_rank"]
)


# If historical market cap exists, add room-to-scale Convex features.
if HAS_MARKET_CAP:
    df["convex_momentum_room_combo"] = (
        df["ret_12m_rank"]
        + df["market_cap_room"]
    )

    df["convex_value_room_combo"] = (
        df["valuation_rank_combo"]
        + df["market_cap_room"]
    )

    df["convex_optionality_room_combo"] = (
        df["vol_12m_rank"]
        + df["market_cap_room"]
    )

    df["convexity_combo"] = (
        df["ret_12m_rank"]
        + df["vol_12m_rank"]
        + df["market_cap_room"]
    )

print("Added shared, Core Alpha, and Convex Alpha composite features")

Added shared, Core Alpha, and Convex Alpha composite features


In [10]:
(df.notnull().sum() / len(df) * 100)

date                           100.000000
ticker                         100.000000
ret_6m                          96.675225
ret_12m                         93.354442
vol_12m                         93.354442
drawdown                       100.000000
pe_ratio                        94.588114
earnings_yield                 100.000000
sector                          99.788689
company                         98.548241
sub_industry                    98.548241
headquarters                    98.548241
date_added                      98.548241
cik                             98.548241
founded                         98.548241
ret_6m_rank                     96.675225
ret_12m_rank                    93.354442
vol_12m_rank                    93.354442
drawdown_rank                  100.000000
low_vol_rank                    93.354442
drawdown_severity_rank         100.000000
earnings_yield_rank            100.000000
ey_rank                        100.000000
pe_ratio_rank                   94

In [11]:
# ======================================
# Clean Features
# ======================================
#
# Replace inf values and neutral-fill rank/composite features.
#
# We keep raw return / valuation NaNs as-is because notebook 03 handles
# final model-specific dropna based on selected feature columns.

df = df.replace([np.inf, -np.inf], np.nan)

# ---- Rank-like columns ----
rank_cols = [
    "ret_6m_rank",
    "ret_12m_rank",
    "vol_12m_rank",
    "drawdown_rank",
    "drawdown_severity_rank",
    "low_vol_rank",
    "earnings_yield_rank",
    "ey_rank",
    "pe_ratio_rank",
    "pe_cheap_rank",
    "sector_earnings_yield_rank",
    "sector_pe_ratio_rank",
    "sector_pe_cheap_rank",
]

if HAS_MARKET_CAP:
    rank_cols += [
        "market_cap_rank",
        "sector_market_cap_rank",
        "market_cap_room",
        "sector_market_cap_room",
    ]

rank_cols = [c for c in rank_cols if c in df.columns]

df[rank_cols] = df[rank_cols].fillna(0.5)


# ---- Composite columns ----
composite_cols = [
    "valuation_rank_combo",
    "sector_valuation_rank_combo",
    "quality_value_combo",
    "quality_value_combo_ranked",
    "momentum_composite",

    # Core
    "core_quality_value_combo",
    "core_stability_combo",
    "core_momentum_quality_combo",
    "core_defensive_value_combo",

    # Convex
    "convex_momentum_value_combo",
    "convex_momentum_vol_combo",
    "convex_rebound_combo",
    "convex_value_rebound_combo",
]

if HAS_MARKET_CAP:
    composite_cols += [
        "core_size_quality_combo",
        "core_stability_size_combo",
        "core_large_cap_value_combo",
        "convex_momentum_room_combo",
        "convex_value_room_combo",
        "convex_optionality_room_combo",
        "convexity_combo",
    ]

composite_cols = [c for c in composite_cols if c in df.columns]

# Neutral-ish fill:
# Most two-rank composites range roughly 0 to 2, so 1.0 is neutral.
# Three-rank composites range roughly 0 to 3, so 1.5 is neutral.
for col in composite_cols:
    if col == "convexity_combo":
        df[col] = df[col].fillna(1.5)
    else:
        df[col] = df[col].fillna(1.0)


# ---- Other engineered features ----
if "momentum_vol_adj" in df.columns:
    df["momentum_vol_adj"] = df["momentum_vol_adj"].fillna(0)

print("Cleaned V4 features")

coverage = (
    df.notnull().sum()
    / len(df)
    * 100
).sort_values()

coverage

Cleaned V4 features


ret_12m                         93.354442
vol_12m                         93.354442
pe_ratio                        94.588114
ret_6m                          96.675225
date_added                      98.548241
founded                         98.548241
cik                             98.548241
headquarters                    98.548241
sub_industry                    98.548241
company                         98.548241
sector                          99.788689
quality_value_combo            100.000000
quality_value_combo_ranked     100.000000
sector_valuation_rank_combo    100.000000
valuation_rank_combo           100.000000
momentum_vol_adj               100.000000
momentum_composite             100.000000
date                           100.000000
core_stability_combo           100.000000
core_momentum_quality_combo    100.000000
sector_pe_cheap_rank           100.000000
core_defensive_value_combo     100.000000
convex_momentum_value_combo    100.000000
convex_momentum_vol_combo      100

In [12]:
df

,date,ticker,ret_6m,ret_12m,vol_12m,drawdown,pe_ratio,earnings_yield,sector,company,...,momentum_vol_adj,momentum_composite,core_quality_value_combo,core_stability_combo,core_momentum_quality_combo,core_defensive_value_combo,convex_momentum_value_combo,convex_momentum_vol_combo,convex_rebound_combo,convex_value_rebound_combo
0,2010-01-04,A,NaN,NaN,NaN,0.000000,26.902868,0.037171,Healthcare,Agilent Technologies,...,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.399881
1,2010-01-05,A,NaN,NaN,NaN,-0.010863,26.902868,0.037171,Healthcare,Agilent Technologies,...,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.705482
2,2010-01-06,A,NaN,NaN,NaN,-0.014377,26.902868,0.037171,Healthcare,Agilent Technologies,...,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.686614
3,2010-01-07,A,NaN,NaN,NaN,-0.015655,26.902868,0.037171,Healthcare,Agilent Technologies,...,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.646520
4,2010-01-08,A,NaN,NaN,NaN,-0.015974,26.902868,0.037171,Healthcare,Agilent Technologies,...,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.592274
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1903823,2025-12-24,ZTS,-0.188985,-0.228600,0.018299,-0.471565,20.328903,0.049191,Healthcare,Zoetis,...,-12.491890,0.070797,1.996560,0.644219,1.311437,1.279579,1.499944,0.558882,0.938493,2.296843
1903824,2025-12-26,ZTS,-0.186188,-0.221406,0.018303,-0.468449,20.328903,0.049191,Healthcare,Zoetis,...,-12.095988,0.071795,1.996560,0.646207,1.312435,1.279579,1.501940,0.560878,0.936504,2.294855
1903825,2025-12-29,ZTS,-0.186653,-0.224929,0.018302,-0.469502,20.328903,0.049191,Healthcare,Zoetis,...,-12.289231,0.076779,1.996560,0.646207,1.317419,1.279579,1.505932,0.564870,0.942481,2.294855
1903826,2025-12-30,ZTS,-0.200488,-0.226137,0.018300,-0.467691,20.328903,0.049191,Healthcare,Zoetis,...,-12.356356,0.075783,1.996560,0.642230,1.316423,1.279579,1.505932,0.564870,0.944465,2.298831


In [13]:
# ======================================
# Save Features V4
# ======================================

base_cols = [
    "date",
    "ticker",

    # V1
    "ret_6m",
    "ret_12m",
    "vol_12m",
    "drawdown",

    # V2
    "pe_ratio",
    "earnings_yield",

    # V3 shared features
    "ret_6m_rank",
    "ret_12m_rank",
    "vol_12m_rank",
    "drawdown_rank",
    "drawdown_severity_rank",
    "low_vol_rank",
    "momentum_vol_adj",
    "momentum_composite",
    "quality_value_combo",

    # V4 ranked valuation features
    "earnings_yield_rank",
    "ey_rank",
    "pe_ratio_rank",
    "pe_cheap_rank",
    "sector_earnings_yield_rank",
    "sector_pe_ratio_rank",
    "sector_pe_cheap_rank",
    "valuation_rank_combo",
    "sector_valuation_rank_combo",
    "quality_value_combo_ranked",

    # metadata
    "sector",
    "sub_industry",
    "company",
]

market_cap_cols = []

if HAS_MARKET_CAP:
    market_cap_cols = [
        "market_cap",
        "log_market_cap",
        "market_cap_rank",
        "sector_market_cap_rank",
        "market_cap_room",
        "sector_market_cap_room",
    ]

core_cols = [
    "core_quality_value_combo",
    "core_stability_combo",
    "core_momentum_quality_combo",
    "core_defensive_value_combo",
]

if HAS_MARKET_CAP:
    core_cols += [
        "core_size_quality_combo",
        "core_stability_size_combo",
        "core_large_cap_value_combo",
    ]

convex_cols = [
    "convex_momentum_value_combo",
    "convex_momentum_vol_combo",
    "convex_rebound_combo",
    "convex_value_rebound_combo",
]

if HAS_MARKET_CAP:
    convex_cols += [
        "convex_momentum_room_combo",
        "convex_value_room_combo",
        "convex_optionality_room_combo",
        "convexity_combo",
    ]

feature_cols = base_cols + market_cap_cols + core_cols + convex_cols

# Keep only columns that exist.
feature_cols = [c for c in feature_cols if c in df.columns]

features = df[feature_cols].copy()

features.to_parquet(FEATURE_FILE_V4, index=False)

print(f"Saved V4 features to: {FEATURE_FILE_V4}")
print("Shape:", features.shape)
print("\nColumns:")
print(features.columns.tolist())

features.head()

Saved V4 features to: /Users/neilyejjey/stock_signal_engine_v1/data/processed/features_v4.parquet
Shape: (1903828, 38)

Columns:
['date', 'ticker', 'ret_6m', 'ret_12m', 'vol_12m', 'drawdown', 'pe_ratio', 'earnings_yield', 'ret_6m_rank', 'ret_12m_rank', 'vol_12m_rank', 'drawdown_rank', 'drawdown_severity_rank', 'low_vol_rank', 'momentum_vol_adj', 'momentum_composite', 'quality_value_combo', 'earnings_yield_rank', 'ey_rank', 'pe_ratio_rank', 'pe_cheap_rank', 'sector_earnings_yield_rank', 'sector_pe_ratio_rank', 'sector_pe_cheap_rank', 'valuation_rank_combo', 'sector_valuation_rank_combo', 'quality_value_combo_ranked', 'sector', 'sub_industry', 'company', 'core_quality_value_combo', 'core_stability_combo', 'core_momentum_quality_combo', 'core_defensive_value_combo', 'convex_momentum_value_combo', 'convex_momentum_vol_combo', 'convex_rebound_combo', 'convex_value_rebound_combo']


,date,ticker,ret_6m,ret_12m,vol_12m,drawdown,pe_ratio,earnings_yield,ret_6m_rank,ret_12m_rank,...,sub_industry,company,core_quality_value_combo,core_stability_combo,core_momentum_quality_combo,core_defensive_value_combo,convex_momentum_value_combo,convex_momentum_vol_combo,convex_rebound_combo,convex_value_rebound_combo
0,2010-01-04,A,NaN,NaN,NaN,0.000000,26.902868,0.037171,0.5,0.5,...,Life Sciences Tools & Services,Agilent Technologies,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.399881
1,2010-01-05,A,NaN,NaN,NaN,-0.010863,26.902868,0.037171,0.5,0.5,...,Life Sciences Tools & Services,Agilent Technologies,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.705482
2,2010-01-06,A,NaN,NaN,NaN,-0.014377,26.902868,0.037171,0.5,0.5,...,Life Sciences Tools & Services,Agilent Technologies,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.686614
3,2010-01-07,A,NaN,NaN,NaN,-0.015655,26.902868,0.037171,0.5,0.5,...,Life Sciences Tools & Services,Agilent Technologies,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.646520
4,2010-01-08,A,NaN,NaN,NaN,-0.015974,26.902868,0.037171,0.5,0.5,...,Life Sciences Tools & Services,Agilent Technologies,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.592274


In [14]:
# ======================================
# V4 Feature Family Summary
# ======================================

shared_feature_cols = [
    "ret_6m",
    "ret_12m",
    "vol_12m",
    "drawdown",
    "pe_ratio",
    "earnings_yield",
    "ret_6m_rank",
    "ret_12m_rank",
    "vol_12m_rank",
    "drawdown_rank",
    "drawdown_severity_rank",
    "low_vol_rank",
    "momentum_vol_adj",
    "momentum_composite",
    "quality_value_combo",
    "earnings_yield_rank",
    "pe_cheap_rank",
    "sector_earnings_yield_rank",
    "sector_pe_cheap_rank",
    "valuation_rank_combo",
    "sector_valuation_rank_combo",
    "quality_value_combo_ranked",
]

size_feature_cols = [
    "market_cap",
    "log_market_cap",
    "market_cap_rank",
    "sector_market_cap_rank",
    "market_cap_room",
    "sector_market_cap_room",
]

core_feature_cols = [
    "core_quality_value_combo",
    "core_stability_combo",
    "core_momentum_quality_combo",
    "core_defensive_value_combo",
    "core_size_quality_combo",
    "core_stability_size_combo",
    "core_large_cap_value_combo",
]

convex_feature_cols = [
    "convex_momentum_value_combo",
    "convex_momentum_vol_combo",
    "convex_rebound_combo",
    "convex_value_rebound_combo",
    "convex_momentum_room_combo",
    "convex_value_room_combo",
    "convex_optionality_room_combo",
    "convexity_combo",
]

feature_families_v4 = {
    "shared": [c for c in shared_feature_cols if c in features.columns],
    "size": [c for c in size_feature_cols if c in features.columns],
    "core_alpha": [c for c in core_feature_cols if c in features.columns],
    "convex_alpha": [c for c in convex_feature_cols if c in features.columns],
}

feature_family_summary = pd.DataFrame([
    {
        "family": family,
        "num_features": len(cols),
        "features": cols,
    }
    for family, cols in feature_families_v4.items()
])

with open(FEATURE_FAMILY_FILE_V4, "w") as f:
    json.dump(feature_families_v4, f, indent=2)

print(f"Saved V4 feature families to: {FEATURE_FAMILY_FILE_V4}")

feature_family_summary

Saved V4 feature families to: /Users/neilyejjey/stock_signal_engine_v1/data/processed/feature_families_v4.json


,family,num_features,features
0,shared,22,"[ret_6m, ret_12m, vol_12m, drawdown, pe_ratio,..."
1,size,0,[]
2,core_alpha,4,"[core_quality_value_combo, core_stability_comb..."
3,convex_alpha,4,"[convex_momentum_value_combo, convex_momentum_..."


In [15]:
features

,date,ticker,ret_6m,ret_12m,vol_12m,drawdown,pe_ratio,earnings_yield,ret_6m_rank,ret_12m_rank,...,sub_industry,company,core_quality_value_combo,core_stability_combo,core_momentum_quality_combo,core_defensive_value_combo,convex_momentum_value_combo,convex_momentum_vol_combo,convex_rebound_combo,convex_value_rebound_combo
0,2010-01-04,A,NaN,NaN,NaN,0.000000,26.902868,0.037171,0.500000,0.500000,...,Life Sciences Tools & Services,Agilent Technologies,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.399881
1,2010-01-05,A,NaN,NaN,NaN,-0.010863,26.902868,0.037171,0.500000,0.500000,...,Life Sciences Tools & Services,Agilent Technologies,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.705482
2,2010-01-06,A,NaN,NaN,NaN,-0.014377,26.902868,0.037171,0.500000,0.500000,...,Life Sciences Tools & Services,Agilent Technologies,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.686614
3,2010-01-07,A,NaN,NaN,NaN,-0.015655,26.902868,0.037171,0.500000,0.500000,...,Life Sciences Tools & Services,Agilent Technologies,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.646520
4,2010-01-08,A,NaN,NaN,NaN,-0.015974,26.902868,0.037171,0.500000,0.500000,...,Life Sciences Tools & Services,Agilent Technologies,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.592274
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1903823,2025-12-24,ZTS,-0.188985,-0.228600,0.018299,-0.471565,20.328903,0.049191,0.061753,0.079840,...,Pharmaceuticals,Zoetis,1.996560,0.644219,1.311437,1.279579,1.499944,0.558882,0.938493,2.296843
1903824,2025-12-26,ZTS,-0.186188,-0.221406,0.018303,-0.468449,20.328903,0.049191,0.061753,0.081836,...,Pharmaceuticals,Zoetis,1.996560,0.646207,1.312435,1.279579,1.501940,0.560878,0.936504,2.294855
1903825,2025-12-29,ZTS,-0.186653,-0.224929,0.018302,-0.469502,20.328903,0.049191,0.067729,0.085828,...,Pharmaceuticals,Zoetis,1.996560,0.646207,1.317419,1.279579,1.505932,0.564870,0.942481,2.294855
1903826,2025-12-30,ZTS,-0.200488,-0.226137,0.018300,-0.467691,20.328903,0.049191,0.065737,0.085828,...,Pharmaceuticals,Zoetis,1.996560,0.642230,1.316423,1.279579,1.505932,0.564870,0.944465,2.298831
